# PSM Memory LOCOMO benchmark on Colab

This notebook installs the published npm packages, downloads the PSM GGUF model through `psm-memory setup`, ingests LOCOMO turns into a SQLite memory DB, and evaluates evidence retrieval.

Recommended Colab runtime: GPU. Start with a small `LIMIT` smoke test before running the full LOCOMO set.

In [ ]:
# Install Node.js 22. Colab's default Node can be older than the package engine range.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Keep model and embedding caches under /content so the paths are predictable.
# Persist benchmark progress and DB checkpoints under Google Drive.
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_DIR = "/content/drive/MyDrive/psm-memory-locomo"
%env PSM_MEMORY_HOME=/content/psm-memory-cache
!mkdir -p /content/psm-memory-work /content/locomo/results "$GDRIVE_DIR"
%cd /content/psm-memory-work
!npm init -y
!npm install @psm-memory/cli@latest @psm-memory/sdk@latest

In [ ]:
# Ensure the published CLI can prepare the default PSM model and embedding model.
!npx psm-memory setup --pretty

In [ ]:
# Clone only for benchmark data and the Colab harness. The benchmark imports @psm-memory/sdk from npm.
!rm -rf /content/PSM
!git clone --depth 1 https://github.com/chkrishna2001/PSM.git /content/PSM
!cp /content/PSM/benchmark/locomo/colab/locomo-benchmark.mjs /content/psm-memory-work/locomo-benchmark.mjs
!ls -lh /content/PSM/benchmark/locomo/data/locomo10.json /content/psm-memory-work/locomo-benchmark.mjs

In [ ]:
# Resumable ingest. Use LIMIT=0 for the full remaining dataset, or a positive number for this run only.
from pathlib import Path
import json
import shutil

LIMIT = 0
BATCH_SIZE = 20
DB = "/content/locomo/results/locomo-psm-memory.db"
MODEL = "/content/psm-memory-cache/psm-memory-qwen-1.5b-q4_k_m.gguf"
DATA = "/content/PSM/benchmark/locomo/data/locomo10.json"
GDRIVE_DIR = globals().get("GDRIVE_DIR", "/content/drive/MyDrive/psm-memory-locomo")
CHECKPOINT_DIR = f"{GDRIVE_DIR}/checkpoints"
PROGRESS = f"{GDRIVE_DIR}/locomo-ingest-progress.json"

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
checkpoint_db = Path(CHECKPOINT_DIR) / Path(DB).name
if checkpoint_db.exists():
    Path(DB).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint_db, DB)
    for suffix in ("-wal", "-shm"):
        sidecar = Path(str(checkpoint_db) + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, str(DB) + suffix)
    print(f"Restored DB checkpoint from {checkpoint_db}")
if Path(PROGRESS).exists():
    progress = json.loads(Path(PROGRESS).read_text())
    print(f"Resuming from record index {progress.get('next_index', 0)} using {PROGRESS}")

!node /content/psm-memory-work/locomo-benchmark.mjs ingest --data "$DATA" --db "$DB" --model "$MODEL" --limit $LIMIT --batch-size $BATCH_SIZE --progress "$PROGRESS" --checkpoint-dir "$CHECKPOINT_DIR"

In [ ]:
# Evaluate evidence retrieval over the memory DB.
OUT = "/content/locomo/results/locomo-results.json"
GDRIVE_DIR = globals().get("GDRIVE_DIR", "/content/drive/MyDrive/psm-memory-locomo")
!node /content/psm-memory-work/locomo-benchmark.mjs evaluate --data "$DATA" --db "$DB" --out "$OUT" --top-k 3
!cp "$OUT" "$GDRIVE_DIR/locomo-results.json"

In [ ]:
# Inspect output artifacts.
!ls -lh /content/locomo/results
!ls -lh "$GDRIVE_DIR" "$GDRIVE_DIR/checkpoints"
!python - <<'PY'
import json
from pathlib import Path
summary = Path('/content/locomo/results/ingest-summary.json')
results = Path('/content/locomo/results/locomo-results.json')
if summary.exists():
    print('ingest summary')
    print(json.dumps(json.loads(summary.read_text()), indent=2)[:4000])
if results.exists():
    print('evaluation summary')
    print(json.dumps(json.loads(results.read_text())['summary'], indent=2))
PY

For a full run, set `LIMIT = 0` in the ingest cell and rerun ingest + evaluate. Ingest now checkpoints the DB and `locomo-ingest-progress.json` to Google Drive every `BATCH_SIZE` records, then resumes from `next_index` after a Colab restart. The full run can take hours depending on Colab GPU availability and `node-llama-cpp` backend selection.